# Guide intelligent du Musée · 01 Données

**Reconnaître le style d'une œuvre et répondre aux questions des visiteurs**

Projet final du cours *Neural Networks & Deep Learning Foundations*.

Membres de l'équipe : Bastien Ragueneau, Colin Pietri, Cristian Larrain.

Le projet tient en six notebooks, à exécuter dans l'ordre. Celui-ci présente le contexte, explore les données et fixe le découpage que tous les autres réutilisent. Chaque chiffre cité est mesuré dans une cellule.

## Contexte

Un musée des beaux-arts français veut un guide intelligent, sur le téléphone des visiteurs et sur des bornes dans les salles :

1. **Reconnaître une œuvre** : le visiteur photographie un tableau, le guide en reconnaît le style et raconte l'histoire du mouvement.
2. **Répondre en langage naturel** : à une question comme « Quel tableau montre une mer déchaînée ? », le guide propose les œuvres de la collection les plus proches.

La direction attend une recommandation claire : les bornes tournent sur des CPU modestes, l'équipe est petite et le budget limité. **Un réseau de neurones vaut-il le coup en production, ou un modèle classique suffit-il ?**

### Feuille de route

| Jalon | Contenu | Notebook | État |
|---|---|---|---|
| Séance 1 | Dépôt, choix de 5 styles, extraction du sous-ensemble WikiArt | `01_donnees` | fait |
| `m1` | Régression logistique et réseau dense de référence, entraîné avec une boucle propre | `02_baseline` | à faire |
| `m2` | Courbes de suivi train / validation et diagnostic des échecs | `03_diagnostic` | à faire |
| `m3` | ResNet pré-entraîné, comparé au réseau dense | `04_resnet` | à faire |
| `m4` | Recherche par question (embeddings de phrases) et comparaison neuronal / classique | `05_questions` | à faire |
| Rendu | Évaluation finale sur le test, recommandation et limites | `06_recommandation` | à faire |

### Protocole

- Découpage train / validation / test décidé avant tout le reste, sans fuite : un même artiste n'apparaît que dans un seul split, et les statistiques de normalisation viennent du train seul. Tous les notebooks utilisent ce même découpage, calculé par `commun.py`.
- Graines fixées. Un gain faible n'est annoncé qu'avec la moyenne et l'écart sur 3 graines.
- Un modèle simple d'abord : il donne le chiffre à battre.
- Un seul changement à la fois, chaque run noté.
- Le test n'est utilisé qu'une fois, à la fin (`06_recommandation`).

## 1. Mise en place

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from commun import (SEED, UNKNOWN_ARTIST, get_device, load_images, load_metadata,
                    normalization_stats, set_seed, split_indices)

set_seed()
DEVICE = get_device()
print(f"PyTorch {torch.__version__}, calculs sur : {DEVICE}")

## 2. Chargement

Sous-ensemble de [`huggan/wikiart`](https://huggingface.co/datasets/huggan/wikiart) extrait par `data/make_wikiart_subset.py` : 5 styles contrastés, 300 œuvres par style, recadrées en 128 × 128 px. `metadata.csv` décrit chaque œuvre, `subset_info.json` garde la commande et la révision du dataset.

Le chargement, le découpage et la normalisation sont dans `commun.py`, partagé par tous les notebooks.

In [ ]:
meta, CLASSES = load_metadata()  # ordre alphabétique = label_idx = ordre d'ImageFolder
N_CLASSES = len(CLASSES)

print(f"{len(meta)} œuvres, {N_CLASSES} styles : {', '.join(CLASSES)}")
meta.head()

In [ ]:
t0 = time.time()
X_all, y_all = load_images(meta)

print(f"X_all : {tuple(X_all.shape)} {X_all.dtype}, y_all : {tuple(y_all.shape)} {y_all.dtype}, "
      f"chargées en {time.time() - t0:.1f} s")

## 3. Exploration

Le script plafonne à 50 le nombre d'œuvres d'un même artiste nommé, mais pas celles de « Unknown Artist ».

In [ ]:
unknown = meta["artist"] == UNKNOWN_ARTIST
summary = meta.assign(inconnu=unknown).groupby("label").agg(
    oeuvres=("path", "size"),
    artistes_nommes=("artist", lambda a: a[a != UNKNOWN_ARTIST].nunique()),
    pct_artiste_inconnu=("inconnu", "mean"),
)
summary["pct_artiste_inconnu"] = (100 * summary["pct_artiste_inconnu"]).round(1)
summary

In [ ]:
# Genres représentés dans chaque style : un style peut se confondre avec un genre dominant.
pd.crosstab(meta["label"], meta["genre"])

In [ ]:
N_SHOW = 6
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(N_CLASSES, N_SHOW, figsize=(1.6 * N_SHOW, 1.6 * N_CLASSES))
for c, name in enumerate(CLASSES):
    for ax, i in zip(axes[c], rng.choice(np.flatnonzero(y_all == c), size=N_SHOW, replace=False)):
        ax.imshow(X_all[i].permute(1, 2, 0))
        ax.set_xticks([])
        ax.set_yticks([])
    axes[c, 0].set_ylabel(name, rotation=0, ha="right", va="center")
fig.suptitle("Six œuvres tirées au hasard par style")
plt.tight_layout()
plt.show()

## 4. Découpage train / validation / test

60 / 20 / 20, stratifié par style. Les œuvres d'un même artiste nommé restent dans un seul split : sinon le modèle pourrait reconnaître le peintre plutôt que le style. Chaque œuvre d'artiste inconnu forme son propre groupe.

In [ ]:
splits = split_indices(meta)  # contrôle aussi la partition et l'absence d'artiste nommé partagé

pd.DataFrame({k: meta.loc[i, "label"].value_counts() for k, i in splits.items()}).loc[CLASSES]

## 5. Statistiques de normalisation

Moyenne et écart-type par canal, calculés sur le train uniquement. Le ResNet utilisera à la place les statistiques d'ImageNet.

In [ ]:
MEAN, STD = normalization_stats(X_all, splits["train"])
print(f"moyenne (R, V, B) : {MEAN.numpy().round(3)}, écart-type : {STD.numpy().round(3)}")